# w3-analysis — Week 3: Word2Vec Embeddings, Cultural Dimensions & WEAT Tests

**Project:** Making Taste Legible: Symbolic Boundaries and Expert Valuation in Whisky Reviews

**Superseded for publication:** This development notebook is retained for provenance. Canonical corrected computations are implemented in `analysis/` and presented in `corrected_results.ipynb`.

Three tasks:
- **Task A:** Train Word2Vec on the whisky review corpus
- **Task B:** Embedding audit + construct 4 cultural dimensions (Kozlowski et al. 2019)
- **Task C:** WEAT association tests (Caliskan et al. 2017)

## 1. Setup & Data Loading

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, re, os, random
from collections import defaultdict
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

os.chdir('/Users/mac/Desktop/CSSS594/FinalProject')
os.makedirs('figures', exist_ok=True)
os.makedirs('models', exist_ok=True)

sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
                      'figure.dpi': 120, 'savefig.dpi': 150, 'savefig.bbox': 'tight'})
PALETTE = sns.color_palette('tab10')

def save_fig(name):
    for ext in ['pdf', 'png']:
        plt.savefig(f'figures/{name}.{ext}', bbox_inches='tight')
    plt.close()

# Load tokenized data
df = pd.read_parquet('data/whiskyfun_tokenized.parquet')
print(f"Loaded {len(df):,} reviews")

# Load dictionary for reference
with open('data/whiskyfun_dictionary_v2.json') as f:
    dictionary = json.load(f)

CAT_KEYS = list(dictionary['categories'].keys())
# V2: derive mapping directly from the dictionary metadata
CAT_SHORT = {key: dictionary['categories'][key]['short_name'] for key in CAT_KEYS}
# Build dict term -> short_name mapping
dict_term_to_cat = {}
for cat_key in CAT_KEYS:
    for term in dictionary['categories'][cat_key]['terms']:
        dict_term_to_cat[term] = CAT_SHORT[cat_key]
print(f"Dictionary: {len(dict_term_to_cat)} unique terms across {len(CAT_KEYS)} categories")

Loaded 11,149 reviews
Dictionary: 247 unique terms across 11 categories


## Task A: Word2Vec Training

### A1. Corpus Preparation

In [52]:
from gensim.models import Word2Vec
from analysis.common import token_sentences

# Prepare corpus: alphabetic tokens only (strip attached punctuation; keep phrase underscores)
print("Preparing corpus...")
sentences = token_sentences(df)
total_tokens = sum(len(s) for s in sentences)

print(f"Total reviews (sentences): {len(sentences):,}")
print(f"Total tokens: {total_tokens:,}")
print(f"Avg tokens/review: {total_tokens/len(sentences):.0f}")

# Vocabulary at min_count=1
all_words = set()
for s in sentences:
    all_words.update(s)
print(f"Vocabulary (min_count=1): {len(all_words):,} unique types")

Preparing corpus...
Total reviews (sentences): 11,149
Total tokens: 1,935,546
Avg tokens/review: 174
Vocabulary (min_count=1): 27,913 unique types


### A2. Train Word2Vec Model

In [53]:
# Train skip-gram model
# dim=150 for stability with small corpus (~1.9M tokens); 10 epochs for better convergence
print("Training Word2Vec (skip-gram, dim=150, window=6, min_count=10, epochs=10)...")
model = Word2Vec(
    sentences,
    sg=1,           # skip-gram
    vector_size=150,
    window=6,
    min_count=10,
    workers=os.cpu_count() - 1 if os.cpu_count() else 3,
    epochs=10,
    seed=42
)

wv = model.wv
vocab = list(wv.key_to_index.keys())
print(f"Vocabulary (min_count=10): {len(vocab):,} words")
print(f"Model training complete.")

# Quick sanity check: most similar to "waxy"
if 'waxy' in wv:
    neighbors = wv.most_similar('waxy', topn=10)
    print(f"\nSanity check — nearest to 'waxy':")
    for word, sim in neighbors:
        print(f"  {word}: {sim:.3f}")
else:
    print("WARNING: 'waxy' not in vocabulary")

Training Word2Vec (skip-gram, dim=150, window=6, min_count=10, epochs=10)...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Vocabulary (min_count=10): 6,383 words
Model training complete.

Sanity check — nearest to 'waxy':
  oily: 0.667
  wax: 0.649
  mineral: 0.646
  persistently: 0.639
  resurgent: 0.625
  thready: 0.625
  stony: 0.618
  waxiness: 0.613
  lemon_scented: 0.612
  sharpish: 0.605


In [54]:
if 'plastic' in wv:
    neighbors = wv.most_similar('plastic', topn=10)
    print(f"\nSanity check — nearest to 'plastic':")
    for word, sim in neighbors:
        print(f"  {word}: {sim:.3f}")
else:
    print("WARNING: 'plastic' not in vocabulary")


Sanity check — nearest to 'plastic':
  leatherette: 0.750
  electronics: 0.660
  chemical: 0.641
  tyre: 0.633
  sneaker: 0.627
  plastic_like: 0.601
  burnt_rubber: 0.590
  verboten: 0.582
  pouch: 0.572
  play_doh: 0.572


### A3. Vocabulary Audit

In [55]:
# Check dictionary terms
dict_terms = list(dict_term_to_cat.keys())
dict_in_vocab = [t for t in dict_terms if t in wv]
dict_missing = [t for t in dict_terms if t not in wv]

print(f"Dictionary terms in embedding vocabulary: {len(dict_in_vocab)} / {len(dict_terms)}")
if dict_missing:
    print(f"Missing ({len(dict_missing)}): {dict_missing}")

# Check dimension pole terms — define them here for audit purposes
# V2 Refined (2026-05-31): D4_Natural_Artificial pole words revised.
dimension_poles = {
    "D1_Quality": ["excellent", "superb", "brilliant", "marvellous", "great", "perfect", "beautiful", "impressive"],
    "D1_Defect": ["poor", "flawed", "weak", "dull", "disappointing", "mediocre", "failed", "unpleasant"],
    "D2_Complex": ["complex", "layered", "deep", "evolving", "sophisticated", "multidimensional", "intricate"],
    "D2_Simple": ["simple", "plain", "basic", "narrow", "monolithic", "monotonous", "monotone"],
    "D3_Balanced": ["balanced", "integrated", "harmonious", "coherent", "precise", "elegant"],
    "D3_Unbalanced": ["unbalanced", "disjointed", "rough", "messy", "clumsy", "excessive", "awkward"],
    "D4_Natural": ["natural", "honest", "earthy", "waxy", "old_school", "traditional", "old_style", "classic", "genuine"],
    "D4_Artificial": ["artificial", "chemical", "plastic", "rubber", "metallic", "solvent", "industrial", "doctored", "bland", "hollow"],
}

# WEAT target/attribute terms
weat_terms_extra = {
    "high_score": ["complex", "waxy", "tropical_fruit", "balanced", "long_finish", "elegant", "rancio", "mineral"],
    "low_score": ["thin", "bitter", "rubbery", "cardboard", "short_finish", "weak", "dull", "simple"],
    "flaws": ["rubber", "sulphur", "cardboard", "soap", "metallic", "feinty", "solvent"],
    "neutral_style": ["barley", "malt", "cereal", "apple", "vanilla", "honey"],
    "weat_quality_a": ["excellent", "superb", "brilliant", "marvellous", "perfect", "impressive"],
    "weat_defect_a": ["poor", "flawed", "disappointing", "mediocre", "failed", "unpleasant"],
    "weat_defect_b": ["poor", "flawed", "dull", "weak", "disappointing"],
    "weat_quality_b": ["excellent", "great", "superb", "brilliant", "marvellous"],
    # WEAT Supplementary Test 3 (V2 Refined 2026-05-31)
    "old_school_X": ["old_school", "old_style", "classic", "traditional", "genuine", "waxy", "honest"],
    "modern_Y": ["doctored", "industrial", "bland", "hollow", "botoxed", "wood_driven", "cask_driven"],
}

print("\nDimension & WEAT pole term audit:")
for pole_name, terms in {**dimension_poles, **weat_terms_extra}.items():
    in_vocab = [t for t in terms if t in wv]
    missing = [t for t in terms if t not in wv]
    status = f"{len(in_vocab)}/{len(terms)} in vocab"
    if missing:
        status += f"  MISSING: {missing}"
    print(f"  {pole_name:<25s}: {status}")

Dictionary terms in embedding vocabulary: 239 / 247
Missing (8): ['black_pepper', 'star_anise', 'white_pepper', 'sea_breeze', 'cream_sherry', 'dried_apricot', 'palo_cortado', 'pedro_ximenez']

Dimension & WEAT pole term audit:
  D1_Quality               : 8/8 in vocab
  D1_Defect                : 7/8 in vocab  MISSING: ['mediocre']
  D2_Complex               : 5/7 in vocab  MISSING: ['sophisticated', 'multidimensional']
  D2_Simple                : 5/7 in vocab  MISSING: ['monotonous', 'monotone']
  D3_Balanced              : 6/6 in vocab
  D3_Unbalanced            : 5/7 in vocab  MISSING: ['messy', 'awkward']
  D4_Natural               : 9/9 in vocab
  D4_Artificial            : 10/10 in vocab
  high_score               : 7/8 in vocab  MISSING: ['long_finish']
  low_score                : 7/8 in vocab  MISSING: ['short_finish']
  flaws                    : 7/7 in vocab
  neutral_style            : 6/6 in vocab
  weat_quality_a           : 6/6 in vocab
  weat_defect_a            : 5/6 

### A4. Save Model

In [56]:
# Save full model and KeyedVectors
model.save('models/w2v_whisky.model')
wv.save('models/w2v_whisky.kv')
print("Saved: models/w2v_whisky.model")
print("Saved: models/w2v_whisky.kv")

Saved: models/w2v_whisky.model
Saved: models/w2v_whisky.kv


## Task B: Embedding Audit & Dimension Construction

### B1. Nearest-Neighbor Audit

In [57]:
# 15 anchor terms for neighbor audit
anchor_terms = [
    "waxy", "rancio", "tropical_fruit", "peat", "medicinal",
    "mineral", "sulphur", "rubber", "cardboard", "sherry",
    "oak", "balanced", "complex", "thin", "short_finish"
]

neighbor_results = []
for anchor in anchor_terms:
    if anchor not in wv:
        print(f"  {anchor}: MISSING from vocabulary — skipping")
        continue
    neighbors = wv.most_similar(anchor, topn=10)
    for rank, (word, sim) in enumerate(neighbors, 1):
        cat = dict_term_to_cat.get(word, '—')
        neighbor_results.append({
            'anchor': anchor, 'rank': rank, 'neighbor': word,
            'similarity': round(sim, 4), 'neighbor_category': cat
        })

neighbor_df = pd.DataFrame(neighbor_results)
neighbor_df.to_csv('data/v2/w3_neighbor_audit.csv', index=False)
print(f"Saved: data/v2/w3_neighbor_audit.csv ({len(neighbor_df)} rows)")

# Display table grouped by anchor
for anchor in [a for a in anchor_terms if a in wv]:
    subset = neighbor_df[neighbor_df['anchor'] == anchor]
    print(f"\n--- {anchor} ---")
    for _, row in subset.iterrows():
        print(f"  {row['rank']:2d}. {row['neighbor']:<25s} {row['similarity']:.4f}  [{row['neighbor_category']}]")

  short_finish: MISSING from vocabulary — skipping
Saved: data/v2/w3_neighbor_audit.csv (140 rows)

--- waxy ---
   1. oily                      0.6667  [texture]
   2. wax                       0.6489  [—]
   3. mineral                   0.6463  [mineral]
   4. persistently              0.6386  [—]
   5. resurgent                 0.6252  [—]
   6. thready                   0.6246  [—]
   7. stony                     0.6184  [—]
   8. waxiness                  0.6127  [—]
   9. lemon_scented             0.6120  [—]
  10. sharpish                  0.6052  [—]

--- rancio ---
   1. bois                      0.6492  [—]
   2. earthen                   0.6322  [—]
   3. fin                       0.6267  [—]
   4. iberico                   0.6152  [—]
   5. mulch                     0.6028  [—]
   6. balsamic                  0.5999  [sherry]
   7. cured                     0.5993  [—]
   8. vors                      0.5953  [—]
   9. dark_fruit                0.5902  [—]
  10. treacle     

### B2. Construct 4 Cultural Dimensions

In [58]:
# Function word filter for top/bottom display
FUNCTION_WORDS = {
    'the', 'and', 'with', 'some', 'this', 'that', 'but', 'not', 'its', 'for',
    'are', 'was', 'has', 'had', 'been', 'were', 'will', 'would', 'could',
    'should', 'may', 'might', 'shall', 'can', 'also', 'still', 'just', 'very',
    'quite', 'more', 'less', 'much', 'many', 'few', 'bit', 'lot', 'well',
    'now', 'then', 'here', 'there', 'than', 'rather', 'indeed', 'even',
    'only', 'really', 'pretty', 'about', 'like', 'back', 'after', 'before',
    'over', 'under', 'between', 'onto', 'into', 'side', 'way', 'time',
    'while', 'though', 'although', 'because', 'since', 'once', 'already',
    'yet', 'ever', 'never', 'always', 'usually', 'often', 'sometimes',
    'around', 'right', 'left', 'top', 'bottom', 'front',
    'all', 'you', 'from', 'have', 'these', 'white', 'which', 'they', 'too',
    'say', 'what', 'wee', 'find', 'slightly', 'same', 'other', 'hint',
    'note', 'touch', 'drop', 'water', 'nose', 'mouth', 'finish', 'palate',
    'comment', 'colour', 'color', 'bottle', 'getting', 'going', 'comes',
    'makes', 'seems', 'feels', 'feeling', 'says', 'think', 'know', 'able',
    'perhaps', 'maybe', 'almost', 'something', 'nothing', 'anything',
    'doesn', 'isn', 'wasn', 'weren', 'aren', 'wouldn', 'couldn',
    'don', 'didn', 'haven', 'hasn', 'need', 'use', 'made', 'making',
    'takes', 'give', 'adds', 'adding', 'let', 'got', 'get', 'put',
    'whole', 'half', 'full', 'issue', 'case', 'point', 'kind', 'sort',
    'year', 'old', 'one', 'two', 'three', 'first', 'second', 'new',
    'big', 'small', 'little', 'high', 'low', 'good', 'great', 'nice',
    'fine', 'best', 'better', 'another', 'others', 'said', 'part', 'end',
    'home', 'days', 'weeks', 'months', 'price', 'buy', 'worth',
}

def compute_dimension(positive_terms, negative_terms, dim_name):
    """Compute a cultural dimension from pole terms. Returns dim_vector and projections."""
    # Filter to terms in vocabulary
    pos_in = [t for t in positive_terms if t in wv]
    neg_in = [t for t in negative_terms if t in wv]

    print(f"\n{dim_name}:")
    print(f"  Positive pole ({len(pos_in)}/{len(positive_terms)} in vocab): {pos_in}")
    missing_pos = [t for t in positive_terms if t not in wv]
    if missing_pos: print(f"    Missing: {missing_pos}")
    print(f"  Negative pole ({len(neg_in)}/{len(negative_terms)} in vocab): {neg_in}")
    missing_neg = [t for t in negative_terms if t not in wv]
    if missing_neg: print(f"    Missing: {missing_neg}")

    if len(pos_in) < 2 or len(neg_in) < 2:
        print(f"  WARNING: Insufficient pole terms — skipping dimension")
        return None, None, None

    # Compute dimension vector
    pos_vec = np.mean([wv[t] for t in pos_in], axis=0)
    neg_vec = np.mean([wv[t] for t in neg_in], axis=0)
    dim_vec = pos_vec - neg_vec
    dim_vec = dim_vec / np.linalg.norm(dim_vec)

    # Project all vocabulary words (content words only for display)
    content_vocab = [w for w in vocab if w not in FUNCTION_WORDS and w.isalpha()]
    projections = {w: float(np.dot(wv[w], dim_vec)) for w in content_vocab}

    # Top and bottom 15 content words
    sorted_words = sorted(projections.items(), key=lambda x: -x[1])
    top15 = sorted_words[:15]
    bottom15 = sorted_words[-15:][::-1]  # most negative first

    print(f"\n  Top 15 (toward {dim_name.split('_')[0]} positive):")
    for w, s in top15:
        cat = dict_term_to_cat.get(w, '—')
        print(f"    {w:<25s} {s:+.4f}  [{cat}]")

    print(f"\n  Bottom 15 (toward negative pole):")
    for w, s in bottom15:
        cat = dict_term_to_cat.get(w, '—')
        print(f"    {w:<25s} {s:+.4f}  [{cat}]")

    return dim_vec, projections, {'pos_in': pos_in, 'neg_in': neg_in, 'top15': top15, 'bottom15': bottom15}

# Define dimension pole terms
# V2 Refined (2026-05-31): Natural_Artificial pole words revised following
# empirical term audit. Removed: clean, pure, organic, subtle (positive);
# synthetic, forced (negative — not in vocabulary). Added: old_style, classic,
# genuine (positive — production tradition); bland, hollow (negative —
# industrial homogeneity). See theoretical_framework.md §3.5 and discussion
# in w3_analysis.ipynb.
dims_config = {
    "D1_Quality_Defect": {
        "positive": ["excellent", "superb", "brilliant", "marvellous", "great", "perfect", "beautiful", "impressive"],
        "negative": ["poor", "flawed", "weak", "dull", "disappointing", "mediocre", "failed", "unpleasant"]
    },
    "D2_Complexity_Simplicity": {
        "positive": ["complex", "layered", "deep", "evolving", "sophisticated", "multidimensional", "intricate"],
        "negative": ["simple", "plain", "basic", "narrow", "monolithic", "straightforward", "monotonous", "monotone"]
    },
    "D3_Balance_Imbalance": {
        "positive": ["balanced", "integrated", "harmonious", "coherent", "precise", "elegant"],
        "negative": ["unbalanced", "disjointed", "rough", "messy", "clumsy", "excessive", "awkward"]
    },
    "D4_Natural_Artificial": {
        "positive": ["natural", "honest", "earthy", "waxy", "old_school", "traditional",
                     "old_style", "classic", "genuine"],
        "negative": ["artificial", "chemical", "plastic", "rubber", "metallic", "solvent",
                     "industrial", "doctored", "bland", "hollow"]
    }
}

dim_vectors = {}
all_dim_projections = {}
dim_metadata = {}

for dim_name, config in dims_config.items():
    dim_vec, projections, meta = compute_dimension(
        config['positive'], config['negative'], dim_name
    )
    if dim_vec is not None:
        dim_vectors[dim_name] = dim_vec
        all_dim_projections[dim_name] = projections
        dim_metadata[dim_name] = meta


D1_Quality_Defect:
  Positive pole (8/8 in vocab): ['excellent', 'superb', 'brilliant', 'marvellous', 'great', 'perfect', 'beautiful', 'impressive']
  Negative pole (7/8 in vocab): ['poor', 'flawed', 'weak', 'dull', 'disappointing', 'failed', 'unpleasant']
    Missing: ['mediocre']

  Top 15 (toward D1 positive):
    exquisite                 +1.4899  [—]
    outstanding               +1.3883  [eval]
    terrific                  +1.2864  [—]
    riddled                   +1.2521  [—]
    stunning                  +1.2286  [—]
    amazing                   +1.1661  [—]
    pristinely                +1.1575  [—]
    beautiful                 +1.1321  [eval]
    superb                    +1.1260  [eval]
    sublime                   +1.1236  [—]
    exceptional               +1.0926  [—]
    astonishing               +1.0877  [—]
    marvellous                +1.0673  [eval]
    immense                   +1.0626  [—]
    gorgeous                  +1.0618  [—]

  Bottom 15 (toward negati

### B3. Project Dictionary Terms onto Dimensions

In [59]:
# Project all in-vocabulary dictionary terms onto each dimension
proj_rows = []
for term, cat in dict_term_to_cat.items():
    if term not in wv:
        continue
    row = {'term': term, 'category': cat}
    for dim_name in dim_vectors:
        short_dim = dim_name.replace('D1_','').replace('D2_','').replace('D3_','').replace('D4_','')
        row[f'{short_dim}_proj'] = round(float(np.dot(wv[term], dim_vectors[dim_name])), 4)
    proj_rows.append(row)

proj_df = pd.DataFrame(proj_rows)
proj_df.to_csv('data/v2/w3_dimension_projections.csv', index=False)
print(f"Saved: data/v2/w3_dimension_projections.csv ({len(proj_df)} terms)")

# Category-level means
dim_cols = [c for c in proj_df.columns if c.endswith('_proj')]
cat_means = proj_df.groupby('category')[dim_cols].mean().round(4)
cat_means.to_csv('data/v2/w3_category_dimension_means.csv')
print(f"Saved: data/v2/w3_category_dimension_means.csv")
print("\n=== Category Dimension Means ===")
print(cat_means.to_string())

Saved: data/v2/w3_dimension_projections.csv (239 terms)
Saved: data/v2/w3_category_dimension_means.csv

=== Category Dimension Means ===
           Quality_Defect_proj  Complexity_Simplicity_proj  Balance_Imbalance_proj  Natural_Artificial_proj
category                                                                                                   
eval                    0.3371                     -0.0272                  0.1977                   0.1729
flaw                   -0.7636                     -0.4477                 -0.8038                  -1.1886
floral                  0.0001                      0.2986                  0.3093                  -0.0829
fruit                   0.1043                      0.0631                  0.2942                  -0.1173
mineral                 0.1328                     -0.2147                  0.0998                  -0.2000
oak                    -0.1956                      0.1156                 -0.0668                  -0.2416

### B4. Figure: Category Dimension Means

In [60]:
cats_ordered = ['fruit', 'peat', 'sherry', 'oak', 'texture', 'mineral', 'flaw', 'structure', 'eval']
dim_labels = ['Quality/Defect', 'Complexity/Simplicity', 'Balance/Imbalance', 'Natural/Artificial']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, dim_col, dim_label in zip(axes.flat, dim_cols, dim_labels):
    values = [cat_means.loc[c, dim_col] if c in cat_means.index else 0 for c in cats_ordered]
    colors = [PALETTE[i % 10] for i in range(len(cats_ordered))]
    bars = ax.bar(range(len(cats_ordered)), values, color=colors, edgecolor='white')
    ax.set_xticks(range(len(cats_ordered)))
    ax.set_xticklabels(cats_ordered, rotation=30, ha='right', fontsize=10)
    ax.set_ylabel('Mean Projection')
    ax.set_title(dim_label)
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002 if val >= 0 else bar.get_height() - 0.008,
                f'{val:+.3f}', ha='center', fontsize=8)

plt.suptitle('Figure W3-1: Dictionary Category Mean Projections onto 4 Cultural Dimensions', fontsize=14)
plt.tight_layout()
save_fig('fig_w3_category_dimensions')

### B5. High-Score vs. Low-Score Descriptor Projections

In [61]:
# Define descriptor groups
high_score_terms = ["complex", "waxy", "tropical_fruit", "balanced", "long_finish", "elegant", "rancio", "mineral"]
low_score_terms = ["thin", "bitter", "rubbery", "cardboard", "short_finish", "weak", "dull", "simple"]

def project_group(terms, group_name):
    """Project terms in group onto all dimensions. Drop missing."""
    in_vocab = [t for t in terms if t in wv]
    missing = [t for t in terms if t not in wv]
    if missing:
        print(f"  {group_name}: MISSING from vocab — {missing}")
    rows = []
    for t in in_vocab:
        row = {'term': t, 'group': group_name}
        for dim_name in dim_vectors:
            short = dim_name.replace('D1_','').replace('D2_','').replace('D3_','').replace('D4_','')
            row[f'{short}_proj'] = round(float(np.dot(wv[t], dim_vectors[dim_name])), 4)
        rows.append(row)
    return pd.DataFrame(rows), in_vocab

high_df, high_terms = project_group(high_score_terms, 'High-Score')
low_df, low_terms = project_group(low_score_terms, 'Low-Score')
hilo_df = pd.concat([high_df, low_df], ignore_index=True)
print(f"\nHigh-score terms in vocab: {len(high_terms)}/{len(high_score_terms)}")
print(f"Low-score terms in vocab: {len(low_terms)}/{len(low_score_terms)}")

# Group means
print("\n=== Group Mean Projections ===")
for group_name, group_df in [('High-Score', high_df), ('Low-Score', low_df)]:
    if len(group_df) == 0: continue
    means = group_df[dim_cols].mean()
    print(f"  {group_name}: " + ", ".join(f"{c}: {means[c]:+.4f}" for c in dim_cols))

# Figure
if len(high_df) > 0 and len(low_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(dim_cols))
    width = 0.35
    high_means = [high_df[c].mean() for c in dim_cols]
    low_means = [low_df[c].mean() for c in dim_cols]

    ax.bar(x - width/2, high_means, width, label='High-Score Descriptors', color=PALETTE[0], edgecolor='white')
    ax.bar(x + width/2, low_means, width, label='Low-Score Descriptors', color=PALETTE[3], edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(['Quality/Defect', 'Complexity/Simplicity', 'Balance/Imbalance', 'Natural/Artificial'], fontsize=10)
    ax.set_ylabel('Mean Projection')
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
    ax.legend()
    ax.set_title('Figure W3-2: High-Score vs. Low-Score Descriptor Projections')
    plt.tight_layout()
    save_fig('fig_w3_hilo_projections')

  High-Score: MISSING from vocab — ['long_finish']
  Low-Score: MISSING from vocab — ['short_finish']

High-score terms in vocab: 7/8
Low-score terms in vocab: 7/8

=== Group Mean Projections ===
  High-Score: Quality_Defect_proj: +0.4198, Complexity_Simplicity_proj: +0.4902, Balance_Imbalance_proj: +0.9189, Natural_Artificial_proj: +0.3429
  Low-Score: Quality_Defect_proj: -0.8744, Complexity_Simplicity_proj: -0.5041, Balance_Imbalance_proj: -0.4337, Natural_Artificial_proj: -0.5917


## Task C: WEAT Association Tests

### C1. WEAT Implementation

In [62]:
def weat_test(X, Y, A, B, n_permutations=10000, seed=42):
    """
    WEAT (Caliskan, Bryson & Narayanan 2017).
    One-sided test: proportion of permutations where test_stat >= observed_stat
    (directional hypothesis: X closer to A than Y is).
    X, Y: target word sets
    A, B: attribute word sets
    """
    random.seed(seed)
    np.random.seed(seed)

    # Filter terms to vocabulary
    X = [t for t in X if t in wv]
    Y = [t for t in Y if t in wv]
    A = [t for t in A if t in wv]
    B = [t for t in B if t in wv]

    if len(X) < 2 or len(Y) < 2 or len(A) < 2 or len(B) < 2:
        return {'effect_size': np.nan, 'p_value': np.nan, 'n_perm': n_permutations,
                'n_X': len(X), 'n_Y': len(Y), 'n_A': len(A), 'n_B': len(B),
                'X_terms': X, 'Y_terms': Y, 'A_terms': A, 'B_terms': B,
                'error': 'Insufficient vocabulary terms'}

    # Association function s(w, A, B)
    def s_word(w, A_set, B_set):
        mean_A = np.mean([np.dot(wv[w], wv[a]) / (np.linalg.norm(wv[w]) * np.linalg.norm(wv[a])) for a in A_set])
        mean_B = np.mean([np.dot(wv[w], wv[b]) / (np.linalg.norm(wv[w]) * np.linalg.norm(wv[b])) for b in B_set])
        return mean_A - mean_B

    # Observed test statistic (mean difference)
    s_X = np.array([s_word(w, A, B) for w in X])
    s_Y = np.array([s_word(w, A, B) for w in Y])
    observed_stat = np.mean(s_X) - np.mean(s_Y)

    # Pooled standard deviation
    all_s = np.concatenate([s_X, s_Y])
    pooled_std = np.std(all_s, ddof=1)
    if pooled_std < 1e-10:
        pooled_std = 1e-10

    # Effect size (Cohen's d)
    effect_size = observed_stat / pooled_std

    # Permutation test (one-sided: stat >= observed)
    all_terms = X + Y
    n_X = len(X)
    count_extreme = 0
    for _ in range(n_permutations):
        random.shuffle(all_terms)
        perm_X = all_terms[:n_X]
        perm_Y = all_terms[n_X:]
        s_perm_X = np.mean([s_word(w, A, B) for w in perm_X])
        s_perm_Y = np.mean([s_word(w, A, B) for w in perm_Y])
        perm_stat = s_perm_X - s_perm_Y
        if perm_stat >= observed_stat:
            count_extreme += 1

    p_value = count_extreme / n_permutations

    return {
        'effect_size': round(effect_size, 4),
        'p_value_one_sided': round(p_value, 4),
        'n_perm': n_permutations,
        'n_X': len(X), 'n_Y': len(Y), 'n_A': len(A), 'n_B': len(B),
        'X_terms': X, 'Y_terms': Y, 'A_terms': A, 'B_terms': B
    }

print("WEAT function defined (one-sided, 10,000 permutations).")

WEAT function defined (one-sided, 10,000 permutations).


### C2. WEAT 1: High-Score vs. Low-Score × Quality/Defect

In [63]:
# WEAT 1
# Note: "weak" and "dull" removed from Attribute B (they appear in Target Y)
# Substituted with "failed" and "unpleasant" (from Dim1 Defect pole)
X1 = ["complex", "waxy", "tropical_fruit", "balanced", "long_finish", "elegant", "rancio", "mineral"]
Y1 = ["thin", "bitter", "rubbery", "cardboard", "short_finish", "weak", "dull", "simple"]
A1 = ["excellent", "superb", "brilliant", "marvellous", "perfect", "impressive"]
B1 = ["poor", "flawed", "disappointing", "mediocre", "failed", "unpleasant"]

print("WEAT 1: High-score vs Low-score descriptors × Quality/Defect")
print(f"  X (high-score, n={len([t for t in X1 if t in wv])} in vocab): {[t for t in X1 if t in wv]}")
missing_x1 = [t for t in X1 if t not in wv]
if missing_x1: print(f"    Missing from vocab: {missing_x1}")
print(f"  Y (low-score, n={len([t for t in Y1 if t in wv])} in vocab): {[t for t in Y1 if t in wv]}")
missing_y1 = [t for t in Y1 if t not in wv]
if missing_y1: print(f"    Missing from vocab: {missing_y1}")

result1 = weat_test(X1, Y1, A1, B1)
print(f"\n  Effect size (d): {result1['effect_size']}")
print(f"  p-value (one-sided): {result1['p_value_one_sided']}")
print(f"  Permutations: {result1['n_perm']:,}")
print(f"  Interpretation: {'High-score descriptors are closer to quality terms (p < 0.05)' if result1['p_value_one_sided'] < 0.05 else 'Not significant'}")

WEAT 1: High-score vs Low-score descriptors × Quality/Defect
  X (high-score, n=7 in vocab): ['complex', 'waxy', 'tropical_fruit', 'balanced', 'elegant', 'rancio', 'mineral']
    Missing from vocab: ['long_finish']
  Y (low-score, n=7 in vocab): ['thin', 'bitter', 'rubbery', 'cardboard', 'weak', 'dull', 'simple']
    Missing from vocab: ['short_finish']

  Effect size (d): 1.6885000467300415
  p-value (one-sided): 0.0003
  Permutations: 10,000
  Interpretation: High-score descriptors are closer to quality terms (p < 0.05)


### C3. WEAT 2: Flaws vs. Neutral × Defect/Quality (+ sulphur sensitivity)

In [64]:
# WEAT 2a: Full (with sulphur)
X2_full = ["rubber", "sulphur", "cardboard", "soap", "metallic", "feinty", "solvent"]
Y2 = ["barley", "malt", "cereal", "apple", "vanilla", "honey"]
A2 = ["poor", "flawed", "dull", "weak", "disappointing"]
B2 = ["excellent", "great", "superb", "brilliant", "marvellous"]

print("WEAT 2 (with sulphur): Flaws vs Neutral × Defect/Quality")
result2_full = weat_test(X2_full, Y2, A2, B2)
print(f"  X flaws (n={result2_full['n_X']}): {result2_full['X_terms']}")
print(f"  Y neutral (n={result2_full['n_Y']}): {result2_full['Y_terms']}")
print(f"  Effect size (d): {result2_full['effect_size']}")
print(f"  p-value (one-sided): {result2_full['p_value_one_sided']}")

# WEAT 2b: Without sulphur
X2_nos = [t for t in X2_full if t != 'sulphur']
print(f"\nWEAT 2 (without sulphur):")
result2_nos = weat_test(X2_nos, Y2, A2, B2)
print(f"  X flaws (n={result2_nos['n_X']}): {result2_nos['X_terms']}")
print(f"  Effect size (d): {result2_nos['effect_size']}")
print(f"  p-value (one-sided): {result2_nos['p_value_one_sided']}")

# Sensitivity comparison
delta_d = result2_full['effect_size'] - result2_nos['effect_size']
print(f"\n  Sensitivity: Δd = {delta_d:+.4f}")
print(f"  {'Sulphur substantially changes effect size (> 0.1 change)' if abs(delta_d) > 0.1 else 'Sulphur does not drive the effect alone — pattern is robust'}")

WEAT 2 (with sulphur): Flaws vs Neutral × Defect/Quality
  X flaws (n=7): ['rubber', 'sulphur', 'cardboard', 'soap', 'metallic', 'feinty', 'solvent']
  Y neutral (n=6): ['barley', 'malt', 'cereal', 'apple', 'vanilla', 'honey']
  Effect size (d): 0.9965000152587891
  p-value (one-sided): 0.034

WEAT 2 (without sulphur):
  X flaws (n=6): ['rubber', 'cardboard', 'soap', 'metallic', 'feinty', 'solvent']
  Effect size (d): 0.9940999746322632
  p-value (one-sided): 0.043

  Sensitivity: Δd = +0.0024
  Sulphur does not drive the effect alone — pattern is robust


### C4. WEAT 3 (Supplementary): Old_School vs Modern/Industrial × Quality/Defect

This supplementary test examines whether production-method vocabulary carries evaluative weight in expert whisky discourse. If expert reviewers valorize traditional craft methods and penalize industrial intervention, old-school terms should differentially associate with quality language while modern/industrial terms should associate with defect language. Test 3 is not part of the original pre-specified primary WEAT design and is interpreted as convergent evidence, not a primary hypothesis test.

In [65]:
# WEAT 3 (Supplementary): Old_School vs Modern/Industrial × Quality/Defect
# X: old-school production descriptors  |  Y: modern/industrial production descriptors
# A: quality terms  |  B: defect terms
# Hypothesis: s(X, A, B) > s(Y, A, B) — old-school terms are closer to quality language.

X3 = ["old_school", "old_style", "classic", "traditional", "genuine", "waxy", "honest"]
Y3 = ["doctored", "industrial", "bland", "hollow", "botoxed", "wood_driven", "cask_driven"]
A3 = ["excellent", "superb", "brilliant", "marvellous", "perfect", "impressive"]
B3 = ["poor", "flawed", "disappointing", "mediocre", "failed", "unpleasant"]

print("WEAT 3 (Supplementary): Old_School vs Modern/Industrial × Quality/Defect")
print(f"  X (old_school, n={len([t for t in X3 if t in wv])} in vocab): {[t for t in X3 if t in wv]}")
missing_x3 = [t for t in X3 if t not in wv]
if missing_x3: print(f"    Missing from vocab: {missing_x3}")
print(f"  Y (modern/industrial, n={len([t for t in Y3 if t in wv])} in vocab): {[t for t in Y3 if t in wv]}")
missing_y3 = [t for t in Y3 if t not in wv]
if missing_y3: print(f"    Missing from vocab: {missing_y3}")

# Filter attribute terms to in-vocabulary (matching weat_test behaviour)
A3_in = [a for a in A3 if a in wv]
B3_in = [b for b in B3 if b in wv]
print(f"  A (quality, n={len(A3_in)}/{len(A3)} in vocab): {A3_in}")
missing_a3 = [a for a in A3 if a not in wv]
if missing_a3: print(f"    Missing A terms: {missing_a3}")
print(f"  B (defect, n={len(B3_in)}/{len(B3)} in vocab): {B3_in}")
missing_b3 = [b for b in B3 if b not in wv]
if missing_b3: print(f"    Missing B terms: {missing_b3}")

result3 = weat_test(X3, Y3, A3, B3)
print(f"\n  Effect size (d): {result3['effect_size']}")
print(f"  p-value (one-sided): {result3['p_value_one_sided']}")
print(f"  Permutations: {result3['n_perm']:,}")
print(f"  Interpretation: {'Old-school descriptors are closer to quality terms (p < 0.05)' if result3['p_value_one_sided'] < 0.05 else 'Not significant at p < 0.05'}")

# Individual term-level diagnostics (using filtered attribute sets)
print(f"\n  Individual s(w, A, B) scores:")
for term in X3 + Y3:
    if term in wv:
        label = "[Old]" if term in X3 else "[Modern]"
        s = np.mean([np.dot(wv[term], wv[a]) / (np.linalg.norm(wv[term]) * np.linalg.norm(wv[a])) for a in A3_in]) \
          - np.mean([np.dot(wv[term], wv[b]) / (np.linalg.norm(wv[term]) * np.linalg.norm(wv[b])) for b in B3_in])
        print(f"    {label} {term:<20s}: {s:+.4f}")
    else:
        print(f"    {'[--]':<7s} {term:<20s}: NOT IN VOCAB")

WEAT 3 (Supplementary): Old_School vs Modern/Industrial × Quality/Defect
  X (old_school, n=7 in vocab): ['old_school', 'old_style', 'classic', 'traditional', 'genuine', 'waxy', 'honest']
  Y (modern/industrial, n=7 in vocab): ['doctored', 'industrial', 'bland', 'hollow', 'botoxed', 'wood_driven', 'cask_driven']
  A (quality, n=6/6 in vocab): ['excellent', 'superb', 'brilliant', 'marvellous', 'perfect', 'impressive']
  B (defect, n=5/6 in vocab): ['poor', 'flawed', 'disappointing', 'failed', 'unpleasant']
    Missing B terms: ['mediocre']

  Effect size (d): 1.625
  p-value (one-sided): 0.0003
  Permutations: 10,000
  Interpretation: Old-school descriptors are closer to quality terms (p < 0.05)

  Individual s(w, A, B) scores:
    [Old] old_school          : +0.1787
    [Old] old_style           : +0.1279
    [Old] classic             : +0.1422
    [Old] traditional         : -0.0535
    [Old] genuine             : +0.0504
    [Old] waxy                : +0.1352
    [Old] honest       

### C4. WEAT Results Table

In [66]:
weat_results = pd.DataFrame([
    {
        'Test': 'WEAT 1: High vs Low × Quality/Defect',
        'EffectSize_d': result1['effect_size'],
        'p_value_one_sided': result1['p_value_one_sided'],
        'N_permutations': result1['n_perm'],
        'n_X': result1['n_X'], 'n_Y': result1['n_Y'],
        'n_A': result1['n_A'], 'n_B': result1['n_B'],
        'analysis_role': 'primary',
        'notes': f"X missing: {[t for t in X1 if t not in wv]}, Y missing: {[t for t in Y1 if t not in wv]}"
    },
    {
        'Test': 'WEAT 2: Flaws vs Neutral × Defect/Quality (with sulphur)',
        'EffectSize_d': result2_full['effect_size'],
        'p_value_one_sided': result2_full['p_value_one_sided'],
        'N_permutations': result2_full['n_perm'],
        'n_X': result2_full['n_X'], 'n_Y': result2_full['n_Y'],
        'n_A': result2_full['n_A'], 'n_B': result2_full['n_B'],
        'analysis_role': 'primary',
        'notes': f"Target X: {result2_full['X_terms']}"
    },
    {
        'Test': 'WEAT 2: Flaws vs Neutral × Defect/Quality (without sulphur)',
        'EffectSize_d': result2_nos['effect_size'],
        'p_value_one_sided': result2_nos['p_value_one_sided'],
        'N_permutations': result2_nos['n_perm'],
        'n_X': result2_nos['n_X'], 'n_Y': result2_nos['n_Y'],
        'n_A': result2_nos['n_A'], 'n_B': result2_nos['n_B'],
        'analysis_role': 'sensitivity',
        'notes': f"Target X: {result2_nos['X_terms']}; Δd from full: {delta_d:+.4f}"
    },
    {
        'Test': 'WEAT 3 (Suppl.): Old_School vs Modern × Quality/Defect',
        'EffectSize_d': result3['effect_size'],
        'p_value_one_sided': result3['p_value_one_sided'],
        'N_permutations': result3['n_perm'],
        'n_X': result3['n_X'], 'n_Y': result3['n_Y'],
        'n_A': result3['n_A'], 'n_B': result3['n_B'],
        'analysis_role': 'supplementary_weat',
        'notes': 'Tests production-method vocabulary evaluative weight; not pre-registered primary'
    }
])

weat_results.to_csv('data/v2/w3_weat_results.csv', index=False)
print("Saved: data/v2/w3_weat_results.csv")
print("\n=== WEAT Results ===")
print(weat_results[['Test', 'EffectSize_d', 'p_value_one_sided', 'analysis_role']].to_string(index=False))

Saved: data/v2/w3_weat_results.csv

=== WEAT Results ===
                                                       Test  EffectSize_d  p_value_one_sided      analysis_role
                       WEAT 1: High vs Low × Quality/Defect        1.6885             0.0003            primary
   WEAT 2: Flaws vs Neutral × Defect/Quality (with sulphur)        0.9965             0.0340            primary
WEAT 2: Flaws vs Neutral × Defect/Quality (without sulphur)        0.9941             0.0430        sensitivity
     WEAT 3 (Suppl.): Old_School vs Modern × Quality/Defect        1.6250             0.0003 supplementary_weat


### C5. WEAT Figures

The WEAT results above are presented in tabular form. The following figures visualize:
- **Figure W3-C1:** Effect sizes across all WEAT tests (forest-plot summary)
- **Figure W3-C2:** Per-term association scores s(w, A, B) for each test, showing which specific words drive each effect

These figures make the WEAT results more interpretable: effect sizes alone don't reveal _which_ terms carry the association signal or whether the effect is driven by one outlier term versus a consistent group-level pattern.

In [67]:
# ------------------------------------------------------------
# C5a. Compute per-term WEAT association scores
# ------------------------------------------------------------
def compute_weat_term_scores(X_terms, Y_terms, A_terms, B_terms):
    """Compute s(w, A, B) for each target term in a WEAT test.
    
    s(w, A, B) = mean cosine similarity of w with all a ∈ A 
               - mean cosine similarity of w with all b ∈ B
    
    Returns DataFrame with columns: term, group ('X' or 'Y'), score
    """
    def s_word(w, A_set, B_set):
        w_vec = wv[w]
        w_norm = np.linalg.norm(w_vec)
        mean_A = np.mean([np.dot(w_vec, wv[a]) / (w_norm * np.linalg.norm(wv[a])) 
                          for a in A_set])
        mean_B = np.mean([np.dot(w_vec, wv[b]) / (w_norm * np.linalg.norm(wv[b])) 
                          for b in B_set])
        return mean_A - mean_B
    
    # Filter attribute terms to in-vocabulary
    A_in = [t for t in A_terms if t in wv]
    B_in = [t for t in B_terms if t in wv]
    
    rows = []
    for term in X_terms:
        if term in wv:
            rows.append({'term': term, 'group': 'X', 'score': s_word(term, A_in, B_in)})
    for term in Y_terms:
        if term in wv:
            rows.append({'term': term, 'group': 'Y', 'score': s_word(term, A_in, B_in)})
    return pd.DataFrame(rows)

# Compute per-term scores for all three WEAT tests
weat1_scores = compute_weat_term_scores(X1, Y1, A1, B1)
weat2_scores = compute_weat_term_scores(X2_full, Y2, A2, B2)
weat3_scores = compute_weat_term_scores(X3, Y3, A3, B3)

# Print summary
for name, df, x_label, y_label in [
    ('WEAT 1', weat1_scores, 'High-score', 'Low-score'),
    ('WEAT 2', weat2_scores, 'Flaws', 'Neutral'),
    ('WEAT 3', weat3_scores, 'Old-school', 'Modern'),
]:
    x_mean = df[df['group']=='X']['score'].mean()
    y_mean = df[df['group']=='Y']['score'].mean()
    print(f"{name}: {x_label} (X) mean s = {x_mean:+.4f}, "
          f"{y_label} (Y) mean s = {y_mean:+.4f}, "
          f"Δ = {x_mean - y_mean:+.4f}, "
          f"n={len(df)} terms ({len(df[df['group']=='X'])}X / {len(df[df['group']=='Y'])}Y)")

# Save per-term scores for reference
all_term_scores = pd.concat([
    weat1_scores.assign(test='WEAT 1: High vs Low × Quality/Defect'),
    weat2_scores.assign(test='WEAT 2: Flaws vs Neutral × Defect/Quality'),
    weat3_scores.assign(test='WEAT 3 (Suppl.): Old School vs Modern × Quality/Defect'),
], ignore_index=True)
all_term_scores.to_csv('data/v2/w3_weat_term_scores.csv', index=False)
print(f"\nSaved: data/v2/w3_weat_term_scores.csv ({len(all_term_scores)} rows)")

WEAT 1: High-score (X) mean s = +0.1607, Low-score (Y) mean s = -0.1088, Δ = +0.2694, n=14 terms (7X / 7Y)
WEAT 2: Flaws (X) mean s = +0.1140, Neutral (Y) mean s = +0.0178, Δ = +0.0963, n=13 terms (7X / 6Y)
WEAT 3: Old-school (X) mean s = +0.0718, Modern (Y) mean s = -0.1618, Δ = +0.2335, n=14 terms (7X / 7Y)

Saved: data/v2/w3_weat_term_scores.csv (41 rows)


In [68]:
# ------------------------------------------------------------
# Figure W3-C1: WEAT Effect Size Summary (Forest Plot)
# ------------------------------------------------------------
# Prepare the 4 test results for display
tests_data = [
    ('WEAT 1\nHigh vs Low ×\nQuality/Defect', 
     result1['effect_size'], result1['p_value_one_sided'], 'primary',
     f"n_X={result1['n_X']}, n_Y={result1['n_Y']}"),
    ('WEAT 2\nFlaws vs Neutral ×\nDefect/Quality', 
     result2_full['effect_size'], result2_full['p_value_one_sided'], 'primary',
     f"n_X={result2_full['n_X']}, n_Y={result2_full['n_Y']}"),
    ('WEAT 2\n(no sulphur)\nSensitivity', 
     result2_nos['effect_size'], result2_nos['p_value_one_sided'], 'sensitivity',
     f"n_X={result2_nos['n_X']}, n_Y={result2_nos['n_Y']}"),
    ('WEAT 3 (Suppl.)\nOld School vs\nModern × Quality/Defect', 
     result3['effect_size'], result3['p_value_one_sided'], 'supplementary',
     f"n_X={result3['n_X']}, n_Y={result3['n_Y']}"),
]

labels = [t[0] for t in tests_data]
d_vals = [t[1] for t in tests_data]
p_vals = [t[2] for t in tests_data]
roles  = [t[3] for t in tests_data]
n_info = [t[4] for t in tests_data]

# Color mapping
role_colors = {
    'primary':       PALETTE[0],   # blue
    'sensitivity':   PALETTE[7],   # gray
    'supplementary': PALETTE[1],   # orange
}
bar_colors = [role_colors[r] for r in roles]

fig, ax = plt.subplots(figsize=(11, 5.5))

y_pos = range(len(labels))
bars = ax.barh(y_pos, d_vals, color=bar_colors, edgecolor='white', height=0.55, 
               alpha=0.85)

# Annotate with d, p-value, and significance stars
for i, (d, p, role, ninfo) in enumerate(zip(d_vals, p_vals, roles, n_info)):
    # Significance stars
    if p < 0.001:
        sig = '★★★'
    elif p < 0.01:
        sig = '★★'
    elif p < 0.05:
        sig = '★'
    else:
        sig = 'n.s.'
    
    # Position text to the right of the bar
    x_pos = d + 0.04
    ax.text(x_pos, i, 
            f'd = {d:.3f}   {sig}\np = {p:.4f}   {ninfo}',
            va='center', fontsize=8.5, fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                      edgecolor='lightgray', alpha=0.85))

# Reference lines
ax.axvline(x=0, color='gray', linestyle='-', linewidth=0.8, zorder=0)
# Convention: small, medium, large effect benchmarks
for benchmark, ls in [(0.2, ':'), (0.5, '--'), (0.8, '--')]:
    ax.axvline(x=benchmark, color='gray', linestyle=ls, linewidth=0.5, 
               alpha=0.4, zorder=0)
# Label benchmarks
ax.text(0.2, -0.55, 'small', fontsize=7, color='gray', ha='center', va='top')
ax.text(0.5, -0.55, 'medium', fontsize=7, color='gray', ha='center', va='top')
ax.text(0.8, -0.55, 'large', fontsize=7, color='gray', ha='center', va='top')

ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=9.5)
ax.set_xlabel("Effect Size (Cohen's d)", fontsize=11)
ax.set_title('Figure W3-C1: WEAT Effect Sizes\n'
             'Expert vocabulary associations with Quality/Defect language',
             fontsize=13, fontweight='bold')
ax.set_xlim(-0.1, max(d_vals) + 0.75)
ax.invert_yaxis()

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=PALETTE[0], alpha=0.85, label='Primary Test'),
    Patch(facecolor=PALETTE[7], alpha=0.85, label='Sensitivity Check'),
    Patch(facecolor=PALETTE[1], alpha=0.85, label='Supplementary Test'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9, 
          framealpha=0.9)

# Annotation explaining the test logic
ax.annotate(
    's(X, A, B) > s(Y, A, B)\nX targets closer to A (Quality/Defect)\nthan Y targets are',
    xy=(0.02, 0.02), xycoords='axes fraction',
    fontsize=7.5, color='gray', style='italic',
    bbox=dict(boxstyle='round', facecolor='whitesmoke', alpha=0.7))

plt.tight_layout()
save_fig('fig_w3_weat_effect_sizes')
print("Saved: figures/fig_w3_weat_effect_sizes.{pdf,png}")

Saved: figures/fig_w3_weat_effect_sizes.{pdf,png}


In [69]:
# ------------------------------------------------------------
# Figure W3-C2: WEAT Term-Level Association Scores
# ------------------------------------------------------------
# Three panels showing individual s(w, A, B) for each target term
# Panel A: WEAT 1 — High vs Low descriptors × Quality/Defect
# Panel B: WEAT 2 — Flaws vs Neutral descriptors × Defect/Quality
# Panel C: WEAT 3 — Old School vs Modern × Quality/Defect

fig, axes = plt.subplots(1, 3, figsize=(20, 7.5))

# --- Panel data configuration ---
panel_config = [
    {
        'ax': axes[0],
        'df': weat1_scores,
        'title': 'WEAT 1: High vs Low Score Descriptors',
        'subtitle': 'A = Quality terms | B = Defect terms\ns > 0 → closer to Quality',
        'x_label_group': 'High-Score (X)',
        'y_label_group': 'Low-Score (Y)',
        'color_x': PALETTE[0],
        'color_y': PALETTE[3],
        'id': 'A',
    },
    {
        'ax': axes[1],
        'df': weat2_scores,
        'title': 'WEAT 2: Flaws vs Neutral Descriptors',
        'subtitle': 'A = Defect terms | B = Quality terms\ns > 0 → closer to Defect',
        'x_label_group': 'Flaws (X)',
        'y_label_group': 'Neutral (Y)',
        'color_x': PALETTE[3],   # red for flaws
        'color_y': PALETTE[2],   # green for neutral
        'id': 'B',
    },
    {
        'ax': axes[2],
        'df': weat3_scores,
        'title': 'WEAT 3 (Suppl.): Old School vs Modern',
        'subtitle': 'A = Quality terms | B = Defect terms\ns > 0 → closer to Quality',
        'x_label_group': 'Old-School (X)',
        'y_label_group': 'Modern (Y)',
        'color_x': PALETTE[0],
        'color_y': PALETTE[3],
        'id': 'C',
    },
]

for cfg in panel_config:
    ax = cfg['ax']
    df = cfg['df'].copy()
    
    # Sort: X group first (descending score), then Y group (ascending score)
    df_x = df[df['group'] == 'X'].sort_values('score', ascending=True)
    df_y = df[df['group'] == 'Y'].sort_values('score', ascending=True)
    df_plot = pd.concat([df_y, df_x])  # Y on bottom, X on top
    
    y_positions = range(len(df_plot))
    colors = [cfg['color_x'] if g == 'X' else cfg['color_y'] 
              for g in df_plot['group']]
    
    # Draw horizontal bars
    bars = ax.barh(y_positions, df_plot['score'], color=colors, 
                   edgecolor='white', height=0.65, alpha=0.85)
    
    # Term labels on y-axis
    ax.set_yticks(y_positions)
    ax.set_yticklabels(df_plot['term'], fontsize=9)
    
    # Reference line at zero
    ax.axvline(x=0, color='gray', linestyle='-', linewidth=0.8, zorder=0)
    
    # Group mean lines
    for group_name, group_color, group_data in [
        (cfg['x_label_group'], cfg['color_x'], df[df['group']=='X']),
        (cfg['y_label_group'], cfg['color_y'], df[df['group']=='Y']),
    ]:
        if len(group_data) > 0:
            mean_val = group_data['score'].mean()
            ax.axvline(x=mean_val, color=group_color, linestyle='--', 
                       linewidth=1.5, alpha=0.6)
            # Small annotation
            ax.text(mean_val, -0.8, 
                    f'{group_name}\nmean={mean_val:+.3f}',
                    fontsize=6.5, color=group_color, ha='center', va='top',
                    fontweight='bold')
    
    # Axis labels and formatting
    ax.set_xlabel('Association Score s(w, A, B)', fontsize=9.5)
    ax.set_title(f"({cfg['id']}) {cfg['title']}", fontsize=10.5, fontweight='bold',
                 loc='left')
    ax.text(0.5, 1.02, cfg['subtitle'], transform=ax.transAxes,
            fontsize=7.5, color='gray', ha='center', va='bottom',
            fontstyle='italic')
    
    # Set symmetric-ish x limits
    abs_max = max(abs(df['score'].min()), abs(df['score'].max())) * 1.25
    ax.set_xlim(-abs_max, abs_max)
    
    # Light horizontal grid
    ax.xaxis.grid(True, alpha=0.2, linestyle=':')
    ax.set_axisbelow(True)

# Main title
fig.suptitle('Figure W3-C2: WEAT Individual Term Association Scores\n'
             'Higher s(w, A, B) = term w is closer in embedding space to attribute set A than to B',
             fontsize=13, fontweight='bold', y=1.03)

plt.tight_layout()
save_fig('fig_w3_weat_term_scores')
print("Saved: figures/fig_w3_weat_term_scores.{pdf,png}")

Saved: figures/fig_w3_weat_term_scores.{pdf,png}


## Summary: Results → Claims

### Claim 1: Expert valuation ≠ generic sentiment
- Neighbor audit shows domain-coherent semantic structure (e.g., `waxy` neighbors `oily`, `mineral`, `resurgent` — textural, not generic-positive)
- Dictionary categories occupy distinct regions of semantic space (B3 category means)

### Claim 2: High value = structure, not just pleasure
- High-score descriptors project toward Complexity and Balance dimensions, not just Quality
- Category means show `structure` category distinct from `eval` in embedding space

### Claim 3: Defects are culturally organized → Natural/Artificial as a dual symbolic boundary
- WEAT 1 confirms high-score descriptors differentially associated with quality pole
- WEAT 2 confirms flaw terms differentially associated with defect pole
- D4 (Natural/Artificial) separates flaw terms from fruit/sherry/texture terms
- **V2 Refined (2026-05-31):** Natural/Artificial pole words revised to capture both
  (a) Douglas purity/contamination (sensory level) and
  (b) traditional-craft / industrial-intervention (production level).
  In expert whisky discourse, production legitimacy is encoded as sensory naturalness.
- Sulphur sensitivity check tests context-dependence

### Claim 4: Scores are textually legitimated
- Structural terms (balance, complexity, integrated) occupy distinct region from explicit evaluation terms
- NMF signal (from Week 2) reflected in embedding geometry

### Claim 5 (Supplementary): Production-method vocabulary carries evaluative weight
- WEAT 3 (supplementary): Old_School vs Modern/Industrial × Quality/Defect
- Tests whether traditional-production vocabulary differentially associates with quality language
- Provides convergent evidence for the production-legitimacy interpretation of D4